In [ ]:
!pip install -q transformers datasets sentencepiece accelerate

import torch
import pandas as pd
import re
import os
from datasets import Dataset
from transformers import (
    T5TokenizerFast,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)

# ============================================================
# CLEAN TEXT
# ============================================================
def clean_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)

    sentences = re.split(r'(?<=\.)\s+', text)
    sentences = [s[:1].upper() + s[1:] if len(s) > 0 else s for s in sentences]
    return " ".join(sentences)


# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv("/kaggle/input/datasets/sbhatti/news-summarization/data.csv")
df.dropna(inplace=True)

df["Content"] = df["Content"].apply(clean_text)
df["Summary"] = df["Summary"].apply(clean_text)

df = df[(df["Content"].str.len() > 40) & (df["Summary"].str.len() > 10)]

df["Content"] = "summarize: " + df["Content"]

df = df.sample(n=30000, random_state=42)

dataset = Dataset.from_pandas(df[["Content", "Summary"]])
dataset = dataset.train_test_split(test_size=0.1)


# ============================================================
# MODEL & TOKENIZER
# ============================================================
tokenizer = T5TokenizerFast.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

MAX_INPUT = 500
MAX_TARGET = 250


# ============================================================
# TOKENIZATION
# ============================================================
def tokenize(batch):
    model_inputs = tokenizer(
        batch["Content"],
        max_length=MAX_INPUT,
        truncation=True
    )

    labels = tokenizer(
        batch["Summary"],
        max_length=MAX_TARGET,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


dataset = dataset.map(tokenize, batched=True, remove_columns=dataset["train"].column_names)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)




In [ ]:
# ============================================================
# TRAINING ARGUMENTS (NO WARNINGS)
# ============================================================
training_args = TrainingArguments(
    output_dir="/kaggle/working/model_output",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=200,
    learning_rate=3e-5,
    weight_decay=0.01,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    report_to="none"
)


# ============================================================
# TRAINER
# ============================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()


# ============================================================
# SAVE MODEL
# ============================================================
SAVE_PATH = "/kaggle/working/final_t5_model"
os.makedirs(SAVE_PATH, exist_ok=True)

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("✅ Model saved at:", SAVE_PATH)


# ============================================================
# INFERENCE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()



In [ ]:

def generate_summary(text):

    cleaned_text = clean_text(text)
    word_count = len(cleaned_text.split())

    input_text = "summarize: " + cleaned_text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_INPUT,
        truncation=True
    ).to(device)

    if word_count >= 300:
        min_summary_length = 100
    elif word_count >= 200:
        min_summary_length = 70
    else:
        min_summary_length = 40

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=MAX_TARGET,
            min_length=min_summary_length,
            num_beams=4,
            length_penalty=1.5,
            no_repeat_ngram_size=3,
        )

    summary = tokenizer.decode(output[0], skip_special_tokens=True)

    sentences = re.split(r'(?<=\.)\s+', summary)
    sentences = [s[:1].upper() + s[1:] if len(s) > 0 else s for s in sentences]

    return " ".join(sentences)


# ============================================================
# TEST
# ============================================================
print(generate_summary("""
Nature is not limited to visible landscapes and wildlife; it also includes invisible processes and cycles that operate continuously. The water cycle circulates moisture through evaporation, condensation, and precipitation. The carbon cycle regulates atmospheric balance through respiration, photosynthesis, and decomposition. Seasonal changes influence migration, reproduction, and plant growth. These cycles function without human intervention and have existed for millions of years, shaping the evolution of life on Earth. Beyond its physical components, the term nature also refers to the inherent character or essence of a person or thing. Many cultures consider nature sacred and emphasize harmony between humans and the environment.
"""))